# LLaMA: Open and Efficient Foundation Language Models

Replication of Touvron, Lavril, Izacard, Martinet, Lachaux, Lacroix, Roziere, Goyal,
Hambro, Azhar, Rodriguez, Joulin, Grave and Lample (2023), *LLaMA: Open and Efficient
Foundation Language Models* (arXiv:2302.13971).

LLaMA is a decoder-only Transformer that adopts a now-standard set of modern building blocks:
RMSNorm pre-normalization, rotary positional embeddings (RoPE) applied to the queries and
keys, the SwiGLU feed-forward activation, and no bias terms. We implement all of these from
scratch in a small character-level language model, train it on a text corpus, and reproduce
the core behavior: the loss falls steadily and the model generates fresh, English-like text.

In [1]:
import math, urllib.request, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
BLOCK, BATCH = 64, 32
def get_batch():
    ix = torch.randint(len(data) - BLOCK - 1, (BATCH,))
    x = torch.stack([data[i:i+BLOCK] for i in ix]); y = torch.stack([data[i+1:i+BLOCK+1] for i in ix])
    return x, y
print("corpus chars:", len(text), "| vocab:", V)

corpus chars: 1115394 | vocab: 65


In [3]:
class RMSNorm(nn.Module):                                  # LLaMA uses RMSNorm, not LayerNorm
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

def apply_rope(x):                                         # rotary positional embedding on (B, H, T, hd)
    B, H, T, hd = x.shape; half = hd // 2
    freqs = 1.0 / (10000 ** (torch.arange(0, half).float() / half))
    ang = torch.arange(T)[:, None].float() * freqs[None, :]        # (T, half)
    cos, sin = ang.cos()[None, None], ang.sin()[None, None]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], -1).flatten(-2)

In [4]:
N_EMB, N_HEAD, N_LAYER = 128, 4, 4
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.h, self.hd = N_HEAD, N_EMB // N_HEAD
        self.wq = nn.Linear(N_EMB, N_EMB, bias=False); self.wk = nn.Linear(N_EMB, N_EMB, bias=False)
        self.wv = nn.Linear(N_EMB, N_EMB, bias=False); self.wo = nn.Linear(N_EMB, N_EMB, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(BLOCK, BLOCK)))
    def forward(self, x):
        B, T, C = x.shape
        sp = lambda t: t.view(B, T, self.h, self.hd).transpose(1, 2)
        q, k, v = apply_rope(sp(self.wq(x))), apply_rope(sp(self.wk(x))), sp(self.wv(x))
        att = (q @ k.transpose(-2, -1)) * self.hd ** -0.5
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        out = (torch.softmax(att, -1) @ v).transpose(1, 2).reshape(B, T, C)
        return self.wo(out)

class SwiGLU(nn.Module):                                   # SwiGLU feed-forward, no bias
    def __init__(self, hidden=4*N_EMB):
        super().__init__()
        self.w1 = nn.Linear(N_EMB, hidden, bias=False); self.w3 = nn.Linear(N_EMB, hidden, bias=False)
        self.w2 = nn.Linear(hidden, N_EMB, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self):
        super().__init__(); self.attn = Attention(); self.ff = SwiGLU()
        self.n1 = RMSNorm(N_EMB); self.n2 = RMSNorm(N_EMB)
    def forward(self, x):
        x = x + self.attn(self.n1(x)); return x + self.ff(self.n2(x))

In [5]:
class LLaMA(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(V, N_EMB)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.norm = RMSNorm(N_EMB); self.head = nn.Linear(N_EMB, V, bias=False)
    def forward(self, idx):
        return self.head(self.norm(self.blocks(self.tok(idx))))   # position comes only from RoPE
    @torch.no_grad()
    def generate(self, idx, n):
        for _ in range(n):
            logits = self(idx[:, -BLOCK:])[:, -1, :]
            idx = torch.cat([idx, torch.multinomial(torch.softmax(logits, -1), 1)], 1)
        return idx

model = LLaMA()
print("parameters:", sum(p.numel() for p in model.parameters()))

parameters: 1066368


In [6]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(4000):
    x, y = get_batch()
    loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1) % 500 == 0: print(f"step {step+1}: loss {loss.item():.3f}")

step 500: loss 1.607


step 1000: loss 1.489


step 1500: loss 1.399


step 2000: loss 1.394


step 2500: loss 1.394


step 3000: loss 1.348


step 3500: loss 1.238


step 4000: loss 1.222


In [7]:
model.eval()
out = model.generate(torch.tensor([[stoi["\n"]]]), 400)[0].tolist()
print("".join(itos[i] for i in out))


Is at the soul, stinded thy doves
For this word it had
deeds for me in articular in the
Duke of pure please your set it prison:
So precious buise roads.

SICINIUS:
Sorrow the amazes,
Whose story follow. Dear humbly is it,
That grace from what truits no refect
Do afteroned o'er his hounds, thus, if I fear,
Do rear, if we kiss me spoke my round,
And harso, be ageder'd him in wonderity mercy.

AUTOLY
